# Data Processing Pipeline

## Purpose

This document tracks the exploration, processing, parsing and ultimate storage pattern of the [Bigfoot Sightings Dataset on kaggle](https://www.kaggle.com/datasets/josephvm/bigfoot-sightings-data)

## Prerequisites

* TODO:

In [1]:
import pandas as pd

df = pd.read_csv('raw/bigfoot-sightings-data.csv')
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5467 entries, 0 to 5466
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Report Type          5467 non-null   str  
 1   Id                   5467 non-null   int64
 2   Class                5016 non-null   str  
 3   Submitted Date       5467 non-null   str  
 4   Headline             5467 non-null   str  
 5   Year                 5015 non-null   str  
 6   Season               5016 non-null   str  
 7   Month                4398 non-null   str  
 8   State                5016 non-null   str  
 9   County               5016 non-null   str  
 10  Location Details     4254 non-null   str  
 11  Nearest Town         4694 non-null   str  
 12  Nearest Road         4310 non-null   str  
 13  Observed             4958 non-null   str  
 14  Also Noticed         3271 non-null   str  
 15  Other Witnesses      4463 non-null   str  
 16  Other Stories        3529 non-null 

### Noticed Patterns

We have 28 columns, with `str` data types dominating the data set.  We need to split these up into their various related entities, starting with the independent (non-foreign-key-bearing) entities before moving on to their dependents.  There is also the task of finding some way to classify thing's as duplicates and account for null values, as this will affect the cardinality of the entity sets we're putting these into.

We can't begin to answer these questions without collecting some basic statistics about the data.

In [3]:
# Basic null analysis
df.isnull().sum().sort_values(ascending=False)

NameError: name 'df' is not defined

In [4]:
# Cardinality (how many unique values in each column)
df.nunique().sort_values(ascending=False)

NameError: name 'df' is not defined

### High-level Observations

* We have some columns (like `Source Url`) with a very high number of nulls
* Other columns have very few distinct values (low cardinality) such as `Class` and `Report Type`

Now we must start cross-referencing this against our schema and see where we land

In [ ]:
# Source 
df[['Media Source',\
    'Source Url',
    'Media Issue']]\
    .rename(columns={\
        'Media Source': 'Name',\
        'Source Url': 'Url',\
        'Media Issue': 'Issue'\
    }).drop_duplicates().shape
# Output: (440, 3)
# There are 440 unique media sources in the dataset

(440, 3)

In [25]:
# Location
df[[
    'State',
    'County',
    'Nearest Town',
    'Nearest Road'
    ]]\
    .drop_duplicates().shape

(4859, 4)

In [ ]:
df[[
    "Environment",
    "Time And Conditions"
]][df['Environment'].notnull() & df['Time And Conditions'].notnull()]\
    .rename(columns={\
        "Environment": "Environment_Description",\
        "Time And Conditions": "Time_And_Conditions"\
            })\
    .drop_duplicates()

,Environment_Description,Time_And_Conditions
0,"In the middle of the woods, in a clearing cove...",Middle of the night. The only light was the he...
1,"A pine forest, with a bog or swamp on the righ...","Started at 11, ended at about 3-3:30. Weather ..."
4,"Lake front,creek spit, gravel and sand, alder ...","Approximately 12:30 pm, partially coudy/sunny."
5,This sighting was located at approximately 1 t...,About 12:00 Midnight / full moon / clear / dim...
6,The forest was made up of mostly birch trees w...,around 6pm
8,"Upland Aspen/birch forest, above O'connor Creek.","Middle of the night, clear weather, thumbnail ..."
12,sighting of track was at the edge of a swamp. ...,Early morning overcast
14,Most of Prince of Wales is forest. There are a...,11:15 PM??
15,There were 2 bridges nearby and we were on a r...,It happend in the daytime and if I was to gues...
16,"Beach on the right side, with brush. A small m...","It was starting to get dark, we needed the lig..."


In [7]:
# Let's create the function that can map a table to an insert statement
def to_sql_insert(table, row, schema: dict):
    columns = []
    values = []
    
    for col in schema.keys():
        columns.append(col)
        values.append(format_value(row[col], schema[col]))
    
    columns_str = ', '.join(columns)
    values_str = ', '.join(values)
    
    return f"INSERT INTO {table} ({columns_str}) VALUES ({values_str});"

## We'll also handle the formatting of values based on their type
def format_value(value, data_type):
    if pd.isnull(value):
        return 'NULL'
    
    if data_type == 'int':
        try:
            return str(int(value))
        except:
            return 'NULL'
    
    elif data_type == 'str':
        safe = str(value).replace("'", "''")  # Escape single quotes
        return f"'{safe}'"

    elif data_type == 'date':
        try:
            date = pd.to_datetime(value).date()
            if pd.isna(date):
                return 'NULL'
            return f"'{date.strftime('%Y-%m-%d')}'"
        except:
            return 'NULL'
        
    else:
        return 'NULL'  # Default fallback for unsupported types

In [2]:
# We have a ton of data, so we may want to extract a small subset of it for the assignment
subset_df =df[df['Report Type'].notna() &
   df['Submitted Date'].notna() &
   df['Author'].notna() &
   df['Media Source'].notna() &
   df['Source Url'].notna() &
   df['Source Url'].str.match(r'^(https?://)?(www\.)?([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}(/.*)?$') &
   df['Media Issue'].notna()]
subset_df.count()

Report Type            112
Id                     112
Class                    0
Submitted Date         112
Headline               112
Year                     0
Season                   0
Month                    0
State                    0
County                   0
Location Details         0
Nearest Town             0
Nearest Road             0
Observed                 0
Also Noticed             0
Other Witnesses          0
Other Stories            0
Time And Conditions      0
Environment              0
Follow-Up                0
Follow-Up Report         0
Date                     0
Author                 112
Media Source           112
Source Url             112
Media Issue            112
Observed.1             108
A & G References         0
dtype: int64

In [75]:
source_df = subset_df[['Media Source',\
    'Source Url',
    'Media Issue']]\
    .rename(columns={\
        'Media Source': 'Name',\
        'Source Url': 'URL',\
        'Media Issue': 'Issue'\
            }).drop_duplicates()\
    .assign(Source_ID=lambda x: range(1, len(x) + 1))\
    [['Source_ID', 'Name', 'URL', 'Issue']]

source_schema = {
    'Source_ID': 'int',
    'Name': 'str',
    'URL': 'str',
    'Issue': 'str'
}
source_inserts = []

source_df

,Source_ID,Name,URL,Issue
3,1,The Delta Discovery,http://www.deltadiscovery.com/story/2013/01/23...,"Volume 15, Issue 4"
33,2,he Clanton Advertiser,http://www.clantonadvertiser.com/articles/2004...,Researches from around the country have contac...
34,3,BC13 Birmingham Tuscaloosa Alabama,http://www.nbc13.com/news/3508750/detail.html,For decades people there have been talking abo...
35,4,The Clanton Advertiser,http://www.clantonadvertiser.com/articles/2004...,It's been almost 25-years since the original r...
50,5,Dispatches from the LP-OP,http://leepeacock2010.blogspot.com/,"According to the witnesses, who spoke to The C..."
...,...,...,...,...
5299,106,travelwisconsin.com,http://www.travelwisconsin.com/article/natural...,This is the signature line to Finding Bigfoot...
5315,107,Baltimore Post-Examiner,http://baltimorepostexaminer.com/big-foot-disc...,Sheriff John Spears told the local press that ...
5316,108,The County Line,http://thecountyline.net/pages/?p=10858,"Of course, Bigfoot sightings have been around ..."
5325,109,Greater Milwaukee Today (www.GMToday.com),http://www.gmtoday.com/news/local_stories/2006...,Witness denies labeling large animal Bigfoot


In [76]:
source_inserts = [
    to_sql_insert('SOURCE', row, source_schema) for _, row in source_df.iterrows()
]

In [77]:
# Now lets make about 20 of these rows for the assignment

with open('source_inserts.sql', 'w') as f:
    for insert in source_inserts[:20]:
        f.write(insert + '\n')
    f.close()
    

In [ ]:
report_schema: dict = {
    'Report_Type': 'str',
    'Headline': 'str',
    'Submitted_Date': 'date',
    'Author': 'str',
    'Source_ID': 'int'
}

# Parse dates into standard format
def parse_date(date_str):
    if pd.isna(date_str):
        return None
    try:
        # Try to parse various date formats
        parsed = pd.to_datetime(date_str, errors='coerce')
        if pd.isna(parsed):
            return None
        return parsed.date()
    except:
        return None

report_df = subset_df[['Report Type', 'Headline', 'Submitted Date', 'Author']]\
    .assign(Source_ID=range(1, len(subset_df) + 1))\
    .rename(columns={\
        'Report Type': 'Report_Type',\
        'Submitted Date': 'Submitted_Date'})\
    .assign(Submitted_Date=lambda x: x['Submitted_Date'].apply(parse_date))\
    .drop_duplicates()
    
report_df.head(2)

,Report_Type,Headline,Submitted_Date,Author,Source_ID
3,Media Article,Legendary Bigfoot sighted near Kasigluk,"Wednesday, January 23, 2013",By KJ Lincoln,1
33,Media Article,Bigfoot sightings of years past draw some inte...,"Sunday, July 11, 2004",T,2


In [5]:
report_inserts = [
    to_sql_insert('REPORT', row, report_schema) for _, row in report_df.iterrows()
]

report_inserts[:20]

NameError: name 'report_df' is not defined

In [6]:
with open('report_inserts.sql', 'w') as f:
    for insert in report_inserts[:20]:
        f.write(insert + '\n')
    f.close()

NameError: name 'report_inserts' is not defined

In [9]:
import re
import random

follow_up = df['Follow-Up'].astype(str).dropna()

investigator_names = []
for text in follow_up:
    text = text.strip()
    match = re.search(r'BFRO Investigator\s+(.+?)\s*$', text, re.IGNORECASE)
    if match:
        investigator_names.append(match.group(1).strip())
        continue

    match = re.search(r'Follow-up report conducted by Investigator\s+(.+?)\.?$', text, re.IGNORECASE)
    if match:
        investigator_names.append(match.group(1).strip())
        continue

    match = re.search(r'Follow-up report by Investigator\s+(.+?)\.?$', text, re.IGNORECASE)
    if match:
        investigator_names.append(match.group(1).strip())
        continue

titles = ['Senior Investigator', 'Lead Investigator', 'Junior Investigator', 'Field Investigator', 'Research Investigator', 'Chief Investigator']
organizations = ['BFRO', 'Cryptozoology Society', 'Wildlife Research Institute', 'Bigfoot Field Research', 'Sighting Documentation Center', 'Cryptid Investigation Bureau']

investigator_df = (
    pd.DataFrame({'Investigator_Name': investigator_names})
    .drop_duplicates()
    .assign(Investigator_ID=lambda x: range(1, len(x) + 1))
    .assign(Title=lambda x: [random.choice(titles) for _ in range(len(x))])
    .assign(Organization=lambda x: [random.choice(organizations) for _ in range(len(x))])
    [['Investigator_ID', 'Investigator_Name', 'Title', 'Organization']]
)

investigator_schema = {
    'Investigator_ID': 'int',
    'Investigator_Name': 'str',
    'Title': 'str',
    'Organization': 'str'
}

investigator_inserts = [
    to_sql_insert('INVESTIGATOR', row, investigator_schema)
    for _, row in investigator_df.iterrows()
]

with open('investigator_inserts.sql', 'w') as f:
    for insert in investigator_inserts[:20]:
        f.write(insert + '\n')


In [3]:
import random

# Build follow-up records using only the reports and investigators that are actually inserted
valid_report_df = report_df.head(20).reset_index(drop=True).copy()
valid_report_df['Report_ID'] = valid_report_df.index + 1
valid_investigator_df = investigator_df.head(20).reset_index(drop=True).copy()
valid_investigator_df['Investigator_ID'] = valid_investigator_df.index + 1

report_notes = (
    subset_df[['Report Type', 'Headline', 'Submitted Date', 'Author', 'Follow-Up Report']]
    .rename(columns={
        'Report Type': 'Report_Type',
        'Submitted Date': 'Submitted_Date'
    })
    .merge(
        valid_report_df[['Report_ID', 'Report_Type', 'Headline', 'Submitted_Date', 'Author']],
        on=['Report_Type', 'Headline', 'Submitted_Date', 'Author'],
        how='inner'
    )
    .dropna(subset=['Follow-Up Report'])
    .drop_duplicates(subset=['Report_ID', 'Follow-Up Report'])
    .reset_index(drop=True)
)

report_notes['Notes'] = report_notes['Follow-Up Report'].astype(str).str.replace('\n', ' ', regex=False).str.strip()
report_notes = report_notes[report_notes['Notes'] != '']

if len(report_notes) == 0:
    report_notes = (
        subset_df[['Follow-Up Report']]
        .dropna()
        .astype(str)
        .str.replace('\n', ' ', regex=False)
        .str.strip()
        .drop_duplicates()
        .reset_index(drop=True)
        .rename(columns={'Follow-Up Report': 'Notes'})
    )
    report_notes = report_notes.assign(
        Report_ID=lambda x: random.choices(list(range(1, min(20, len(report_df)) + 1)), k=len(x))
    )

random.seed(42)
if len(report_notes) > 20:
    report_notes = report_notes.sample(20, random_state=42).reset_index(drop=True)

investigator_ids = list(range(1, min(20, len(valid_investigator_df)) + 1))

follow_up_rows = []
for _, row in report_notes.iterrows():
    report_id = int(row['Report_ID'])
    investigator_id = random.choice(investigator_ids)
    submitted = pd.to_datetime(row['Submitted_Date'], errors='coerce') if 'Submitted_Date' in row else pd.NaT
    if not pd.isna(submitted):
        follow_up_date = submitted + pd.Timedelta(days=random.randint(7, 60))
    else:
        follow_up_date = pd.Timestamp('2000-01-01') + pd.Timedelta(days=random.randint(0, 3650))
    follow_up_rows.append({
        'Report_ID': report_id,
        'Investigator_ID': investigator_id,
        'Follow_Up_Date': follow_up_date,
        'Notes': row['Notes']
    })

follow_up_df = pd.DataFrame(follow_up_rows).head(20)

follow_up_schema = {
    'Report_ID': 'int',
    'Investigator_ID': 'int',
    'Follow_Up_Date': 'date',
    'Notes': 'str'
}

follow_up_inserts = [
    to_sql_insert('FOLLOW_UP', row, follow_up_schema)
    for _, row in follow_up_df.iterrows()
]

with open('follow_up_inserts.sql', 'w') as f:
    for insert in follow_up_inserts:
        f.write(insert + '\n')


NameError: name 'report_df' is not defined

In [4]:
import re
import random

environment_df = df[[
    'Environment',
    'Season',
    'Year',
    'Month',
    'Date',
    'Time And Conditions'
]].rename(columns={
    'Environment': 'Environment_Description',
    'Date': 'Date_Text',
    'Time And Conditions': 'Time_And_Conditions'
})

month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}

def parse_year(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    match = re.search(r'(\d{4})', text)
    if match:
        return int(match.group(1))
    match = re.search(r'(\d{2})$', text)
    if match:
        year = int(match.group(1))
        return 1900 + year if year > 30 else 2000 + year
    return None

def parse_month(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text.isdigit():
        month = int(text)
        return month if 1 <= month <= 12 else None
    return month_map.get(text.capitalize())

def truncate_text(value, max_len):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if len(text) <= max_len else text[:max_len]

environment_df['Year'] = environment_df['Year'].apply(parse_year)
environment_df['Month'] = environment_df['Month'].apply(parse_month)
environment_df = environment_df.assign(
    Year=lambda x: x['Year'].fillna(2000).astype(int),
    Month=lambda x: x['Month'].fillna(1).astype(int)
)

environment_df = environment_df.dropna(subset=['Environment_Description'])
environment_df = environment_df.drop_duplicates()

environment_df['Season'] = environment_df['Season'].apply(lambda v: truncate_text(v, 20))
environment_df['Date_Text'] = environment_df['Date_Text'].apply(lambda v: truncate_text(v, 20))
environment_df['Time_And_Conditions'] = environment_df['Time_And_Conditions'].apply(lambda v: truncate_text(v, 100))

random.seed(42)
if len(environment_df) > 20:
    environment_df = environment_df.sample(20, random_state=42)

environment_df = environment_df.reset_index(drop=True).assign(
    Environment_ID=lambda x: range(1, len(x) + 1)
)[[
    'Environment_ID',
    'Environment_Description',
    'Season',
    'Year',
    'Month',
    'Date_Text',
    'Time_And_Conditions'
]]

environment_schema = {
    'Environment_ID': 'int',
    'Environment_Description': 'str',
    'Season': 'str',
    'Year': 'int',
    'Month': 'int',
    'Date_Text': 'str',
    'Time_And_Conditions': 'str'
}

environment_inserts = [
    to_sql_insert('ENVIRONMENT', row, environment_schema)
    for _, row in environment_df.iterrows()
]

with open('environment_inserts.sql', 'w') as f:
    for insert in environment_inserts:
        f.write(insert + '\n')


NameError: name 'to_sql_insert' is not defined

In [12]:
import random

# Build EVENT records from rows with Class and Observed data
# Match report IDs to the first 20 inserted reports
valid_report_df = report_df.head(20).reset_index(drop=True).copy()
valid_report_df['Report_ID'] = valid_report_df.index + 1

valid_environment_df = environment_df.head(20).reset_index(drop=True).copy()
valid_environment_df['Environment_ID'] = valid_environment_df.index + 1

# Extract event data from source
event_data = df[[
    'Class',
    'Observed',
    'Other Witnesses',
    'Other Stories',
    'Also Noticed'
]].dropna(subset=['Class', 'Observed']).copy()

# Rename columns
event_data = event_data.rename(columns={
    'Class': 'Event_Type',
    'Observed': 'Observed_Description',
    'Other Witnesses': 'Other_Witnesses',
    'Other Stories': 'Witness_Other_Stories',
    'Also Noticed': 'Also_Noticed'
})

# Reset index to align with assignment
event_data = event_data.reset_index(drop=True)

# Assign Report_ID and Environment_ID
# Cycle through valid IDs if there are fewer inserts than data
report_ids = list(range(1, 21))
environment_ids = list(range(1, 21))

event_data['Report_ID'] = event_data.index.map(lambda i: report_ids[i % len(report_ids)])
event_data['Environment_ID'] = event_data.index.map(lambda i: environment_ids[i % len(environment_ids)])
event_data['Location_ID'] = None

# Clean up text fields (truncate for schema safety)
def truncate_text(value, max_len):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if len(text) <= max_len else text[:max_len]

event_data['Event_Type'] = event_data['Event_Type'].apply(lambda v: truncate_text(v, 50) if not pd.isna(v) else None)
event_data['Observed_Description'] = event_data['Observed_Description'].apply(lambda v: truncate_text(v, 5000))
event_data['Other_Witnesses'] = event_data['Other_Witnesses'].apply(lambda v: truncate_text(v, 5000) if not pd.isna(v) else None)
event_data['Witness_Other_Stories'] = event_data['Witness_Other_Stories'].apply(lambda v: truncate_text(v, 5000) if not pd.isna(v) else None)
event_data['Also_Noticed'] = event_data['Also_Noticed'].apply(lambda v: truncate_text(v, 5000) if not pd.isna(v) else None)

# Filter to first 20 and required columns
event_df = event_data.head(20)[[
    'Event_Type',
    'Observed_Description',
    'Other_Witnesses',
    'Witness_Other_Stories',
    'Also_Noticed',
    'Report_ID',
    'Environment_ID',
    'Location_ID'
]].copy()

# Reset index and assign Event_ID
event_df = event_df.reset_index(drop=True).assign(
    Event_ID=lambda x: range(1, len(x) + 1)
)

# Reorder columns
event_df = event_df[[
    'Event_ID',
    'Event_Type',
    'Observed_Description',
    'Other_Witnesses',
    'Witness_Other_Stories',
    'Also_Noticed',
    'Report_ID',
    'Environment_ID',
    'Location_ID'
]]

event_schema = {
    'Event_ID': 'int',
    'Event_Type': 'str',
    'Observed_Description': 'str',
    'Other_Witnesses': 'str',
    'Witness_Other_Stories': 'str',
    'Also_Noticed': 'str',
    'Report_ID': 'int',
    'Environment_ID': 'int',
    'Location_ID': 'int'
}

event_inserts = [
    to_sql_insert('EVENT', row, event_schema)
    for _, row in event_df.iterrows()
]

with open('event_inserts.sql', 'w') as f:
    for insert in event_inserts:
        f.write(insert + '\n')

print(f"Generated {len(event_inserts)} EVENT inserts")

Generated 20 EVENT inserts
